# SupplyMind v3.0-arcadia — Colab Quickstart

Runs the **lightweight** parts of SupplyMind v3 in a Colab notebook:
- R5 Granite RAG retrieval with mxbai-embed-large
- R6 Provider GNN arrival-time regression (pure PyTorch)
- R6 Aqua Regia per-horizon conformal demo

**Skipped in Colab** (too heavy for free tier):
- R4 Dangerous (needs 4 × 9GB LLMs via Ollama)
- R3 Past Self full stack (needs TimesFM-2 2GB ckpt)
- R6 Gethsemane RL training (needs sb3-contrib + 3h on RTX 4080)

Full reproducibility: clone the repo and run `pytest tests/ -q`.

GitHub: https://github.com/ShAuRyA-Noodle/Sleep-Token · Tag: `v3.0-arcadia`

## 1. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/ShAuRyA-Noodle/Sleep-Token.git supplymind
%cd supplymind

# Install slim deps (skip the RL + 150GB model download)
!pip install -q fastapi uvicorn pydantic numpy requests scipy statsmodels sentence-transformers torch onnxruntime

## 2. R5 Granite RAG (retrieval-only, no LLM)

In [ ]:
# Uses the cached 6,483-chunk corpus + mxbai embeddings from the repo
import json, pickle, numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

CKPT = Path('versions/v3_arcadia/checkpoints/granite')

# Download the corpus chunks from GitHub release (if not in cloned repo)
import urllib.request
if not (CKPT / 'corpus_chunks.pkl').exists():
    urllib.request.urlretrieve(
        'https://github.com/ShAuRyA-Noodle/Sleep-Token/raw/main/versions/v3_arcadia/checkpoints/granite/corpus_chunks.pkl',
        CKPT / 'corpus_chunks.pkl')

with open(CKPT / 'corpus_chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)
print(f'Loaded {len(chunks)} chunks')

# Download mxbai from HF hub directly (small, ~1.3 GB)
emb = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')

# Re-embed corpus (takes ~2 min on Colab GPU)
texts = [c['text'] for c in chunks]
corpus_emb = emb.encode(texts, normalize_embeddings=True, batch_size=16, show_progress_bar=True)
print(f'Corpus embedded: {corpus_emb.shape}')

In [ ]:
# Live query demo
def rag_query(q, top_k=5):
    q_emb = emb.encode(q, normalize_embeddings=True)
    sims = corpus_emb @ q_emb
    idx = np.argsort(sims)[::-1][:top_k]
    return [{'doc_id': chunks[i]['doc_id'], 'score': float(sims[i]), 'text': chunks[i]['text'][:200]} for i in idx]

for q in [
    'What was the magnitude of the 2011 Tohoku earthquake?',
    'How long was the Suez Canal blocked in 2021?',
    'Which foundry produces most advanced logic chips?',
]:
    print(f'\nQuery: {q}')
    for hit in rag_query(q, top_k=3):
        print(f"  [{hit['score']:.3f}] {hit['doc_id']}: {hit['text'][:100]}...")

## 3. R6 Provider GNN (pure PyTorch disruption propagation)

In [ ]:
# The custom GCN in pure PyTorch — no torch_geometric
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(2 * in_dim, out_dim)
    def forward(self, x, edge_index):
        n = x.size(0)
        src, dst = edge_index
        agg = torch.zeros_like(x)
        count = torch.zeros(n, 1, device=x.device)
        agg.index_add_(0, src, x[dst])
        count.index_add_(0, src, torch.ones(src.size(0), 1, device=x.device))
        agg = agg / count.clamp(min=1.0)
        return self.lin(torch.cat([x, agg], dim=1))

# Load the hard supply-chain graph (40 nodes)
g = json.loads(Path('server/data/graphs/hard_graph.json').read_text())
print(f'Hard graph: {len(g["nodes"])} nodes, {len(g["edges"])} edges')
print(f'Sample node: {g["nodes"][0]["id"]} ({g["nodes"][0]["name"]}, tier {g["nodes"][0]["tier"]})')

## 4. R6 Aqua Regia per-horizon conformal demo

In [ ]:
import numpy as np

def per_horizon_conformal_band(cal_residuals, alpha):
    n, H = cal_residuals.shape
    q_hat = np.zeros(H)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    k = min(k, n)
    for h in range(H):
        q_hat[h] = float(np.sort(np.abs(cal_residuals[:, h]))[k - 1])
    return q_hat

# Synthetic demo: 30 calibration folds, horizon 14, growing residual magnitude
rng = np.random.default_rng(0)
cal = rng.normal(0, 1, (30, 14)) * np.linspace(1, 3, 14)  # std grows with horizon

q_ph = per_horizon_conformal_band(cal, alpha=0.1)  # 90% nominal
q_pool = np.sort(np.abs(cal).flatten())[int(np.ceil(cal.size * 0.9)) - 1]

print(f'Per-horizon q_hat: {q_ph.round(2)}')
print(f'Pooled q_hat:     {q_pool:.2f} (single value across all steps)')
print()
print('Per-horizon adapts to growing residuals; pooled over-covers early steps, under-covers late.')

## 5. What's next

- Clone the full repo: `git clone https://github.com/ShAuRyA-Noodle/Sleep-Token.git`
- Read [`MODEL_CARD.md`](https://github.com/ShAuRyA-Noodle/Sleep-Token/blob/main/docs/v3/MODEL_CARD.md) for the unified card
- Read [`BENCHMARKS_VS_PUBLIC.md`](https://github.com/ShAuRyA-Noodle/Sleep-Token/blob/main/docs/v3/BENCHMARKS_VS_PUBLIC.md) for comparison to M5/MTEB/MuJoCo
- Read [`PYTORCH_STORY.md`](https://github.com/ShAuRyA-Noodle/Sleep-Token/blob/main/docs/v3/PYTORCH_STORY.md) for the PyTorch-specific engineering story
- Try the live API: https://huggingface.co/spaces/Shaurya-Noodle/Supplymind (v3 Damocles endpoints)
- Watch the 3-min demo: [link in GitHub Release v3.0-arcadia]

*Every negative finding is documented with a world-class follow-up fix in [`FAILURE_TABLE.md`](https://github.com/ShAuRyA-Noodle/Sleep-Token/blob/main/FAILURE_TABLE.md).*

Sleep Token's "Even In Arcadia" (2025) provides the phase-naming theme. Every commit is tagged with a track name.